# Lots Dimension — Gold Layer (`company_db.gold.dim_lot`)

This notebook builds and maintains the **Lots Dimension** table in the Gold layer of the Company data warehouse. It is the canonical reference for every auction lot — capturing item details, format, provenance, category, and auction dates.

---

## What This Notebook Does

1. Extracts lot records from the Silver layer by joining auction results with the auction sales report
2. Assigns or preserves surrogate keys (`lot_dim_key`) using a load-control flag
3. Merges the result into the Gold dimension table (insert new lots, update existing ones)

---

## Source Tables (Silver Layer)

| Table | Description |
|---|---|
| `company_db.silver.auctionresults` | Lot-level auction results — one row per lot per auction |
| `company_db.silver.auction_sales_report` | Auction-level metadata including `auction_category` (joined on `auction_id`) |

## Target Table (Gold Layer)

`company_db.gold.dim_lot`  
Stored in Azure Data Lake Storage Gen2: `abfss://auction@company.dfs.core.windows.net/gold/dim_lot`

---

## MERGE Key

Records are matched on the **3-column composite key**: `auction_id + lot_number + start_date`  
This uniquely identifies a lot within a given auction run.

---

## Load Strategy (controlled by the `flag` widget)

| `flag` value | Behavior |
|---|---|
| `0` | **Initial / Full Load** — surrogate key sequence starts at 1 |
| `1` | **Incremental Load** — surrogate key sequence continues from current `MAX(lot_dim_key)` |

---

## Column Reference

| Column | Type | Description |
|---|---|---|
| `lot_dim_key` | BIGINT | **Surrogate key** — auto-incrementing warehouse identifier |
| `auction_id` | INTEGER | Auction identifier |
| `lot_id` | DOUBLE | Unique lot identifier |
| `lot_number` | DOUBLE | Lot number within the auction catalogue |
| `item_name` | STRING | Full name / description of the lot |
| `link` | STRING | URL link to the lot listing |
| `exportedimgs` | STRING | Exported image file paths for the lot |
| `format` | STRING | Packaging format or size (e.g. `Small`, `Large`) |
| `attribute:country` | STRING | Country attribute tag from the lot metadata |
| `attribute:region` | STRING | Region attribute tag from the lot metadata |
| `region` | STRING | Region of origin |
| `country` | STRING | Country of origin |
| `item_year` | STRING | Item year |
| `lot_category` | STRING | Category classification of the lot |
| `status` | STRING | Lot status (e.g. `Sold`, `Passed`) |
| `start_date` | DATE | Auction start date |
| `end_date` | DATE | Auction end date |
| `auction_category` | STRING | Category of the parent auction |
| `created_at` | TIMESTAMP | Timestamp when the row was first inserted |
| `updated_at` | TIMESTAMP | Timestamp of the last MERGE update |

In [ ]:
catalog = dbutils.widgets.get("catalog")
container = dbutils.widgets.get("container")
storage_account = dbutils.widgets.get("storage_account")

In [ ]:
%run "../setup/002_Environment_variables"

In [ ]:
from pyspark.sql import functions as F

---
## Step 1 · Extract Lots from Silver Layer

Joins `company_db.silver.auctionresults` with `company_db.silver.auction_sales_report` on `auction_id` (INNER JOIN) to enrich lot records with the parent auction’s `auction_category`.

Selects the full set of lot attributes needed for the dimension: item details (`item_name`, `item_year`, `format`, `country`, `region`), lot metadata (`lot_id`, `lot_number`, `lot_category`, `status`, `link`, `exportedimgs`), auction dates (`start_date`, `end_date`), and both attribute tag columns (`attribute:country`, `attribute:region`).

In [ ]:
lot_table = spark.sql(f"""
SELECT 
ar.auction_id, 
ar.lot_id,
ar.lot_number,
ar.item_name,
ar.link,
ar.exportedimgs,
ar.format,
ar.`attribute:country`,
ar.`attribute:region`,
ar.region,
ar.country,
ar.item_year,
ar.lot_category,
ar.status,
ar.start_date,
ar.end_date,
asr.auction_category

FROM
{catalog}.silver.auctionresults as ar
INNER JOIN {catalog}.silver.auction_sales_report asr
On ar.auction_id = asr.auction_id
"""
)


---
## Step 2 · Load Control Flag

The `flag` widget controls the load mode for this run:
- `flag = 0` — **Initial / Full Load.** Surrogate key counter (`lot_dim_key`) starts at 1. Use this on the first ever run or after a full table reset.
- `flag = 1` — **Incremental Load.** Surrogate key counter continues from `MAX(lot_dim_key)` in the existing table. Use this for every scheduled run.

In [ ]:
# flag =  dbutils.widgets.text('flag','0')

In [ ]:
# flag = dbutils.widgets.get('flag')
# flag

---
## Step 3 · Initialize Gold Dimension Table

Creates `company_db.gold.dim_lot` as a **Delta table** if it does not already exist. On subsequent runs this is a no-op — no existing data is affected.

Granularity: **one row per `auction_id` × `lot_number` × `start_date`** (enforced by the MERGE key in Step 6).

The table is backed by ADLS Gen2 at `gold/dim_lot` and supports full ACID transactions (MERGE, time travel, schema evolution).

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.gold.lots_dim
(
  lot_dim_key BIGINT,
  auction_id INTEGER,
  lot_id DOUBLE,
  lot_number DOUBLE,
  item_name STRING,
  link STRING,
  exportedimgs STRING,
  format STRING,
  `attribute:country` STRING,
  `attribute:region` STRING,
  region STRING,
  country STRING,
  item_year STRING,
  lot_category STRING,
  status STRING,
  start_date DATE,
  end_date DATE,
  auction_category STRING,
  created_at TIMESTAMP,
  updated_at TIMESTAMP
)
USING DELTA
LOCATION 'abfss://{container}@{storage_account}.dfs.core.windows.net/gold/lots_dim';""")

In [ ]:
dim_lot = spark.table(f'{catalog}.gold.lots_dim')

---
## Step 4 · Identify New vs Existing Lots

Loads the current `company_db.gold.dim_lot` and LEFT JOINs it against the incoming Silver lots on the **3-column composite key** (`auction_id + lot_number + start_date`).

After the join, records are split into two groups:

| Group | Condition | Next Action |
|---|---|---|
| `new_lots` | `lot_dim_key IS NULL` | Needs a surrogate key assigned, then INSERTed |
| `old_lots` | `lot_dim_key IS NOT NULL` | Surrogate key already exists, will be UPDATEd via MERGE |

In [ ]:
lot_table = lot_table.join(
    dim_lot,
    (dim_lot.auction_id == lot_table.auction_id) &
    (dim_lot.lot_number == lot_table.lot_number) &
    (dim_lot.start_date == lot_table.start_date),
    'left'
).select(
    lot_table.auction_id,
    lot_table.lot_id,
    lot_table.lot_number,
    lot_table.item_name,
    lot_table.link,
    lot_table.exportedimgs,
    lot_table.format,
    lot_table["attribute:country"],
    lot_table["attribute:region"],
    lot_table.region,
    lot_table.country,
    lot_table.item_year,
    lot_table.lot_category,
    lot_table.status,
    lot_table.start_date,
    lot_table.end_date,
    lot_table.auction_category,
    dim_lot.lot_dim_key
).sort('auction_id', 'lot_id')

In [ ]:
new_lots= lot_table.filter(lot_table.lot_dim_key.isNull())
old_lots = lot_table.filter(lot_table.lot_dim_key.isNotNull())

---
## Step 5 · Surrogate Key Assignment for New Lots

Assigns a unique `lot_dim_key` to each new lot using `ROW_NUMBER()` ordered by `start_date` then `lot_number` (earliest auction / lowest lot number gets the lowest key).

The key is offset by `max_value` to avoid collisions with existing records:
```
lot_dim_key = ROW_NUMBER() OVER (ORDER BY start_date, lot_number) + max_value
```

- On an **initial load** (`flag=0`): `max_value = 0`, so keys start at 1.
- On an **incremental load** (`flag=1`): `max_value = MAX(lot_dim_key)` from the existing table, so keys continue seamlessly.

In [ ]:
#  #fetching max_value from existing table
# if flag == '0':
#     max_value = 0
# else:
#     max_value = spark.sql(""" SELECT MAX(lot_dim_key) FROM company_db.gold.lots_dim """).collect()[0][0]

max_value = spark.sql(f"""
    SELECT COALESCE(MAX(lot_dim_key), 0) FROM {catalog}.gold.lots_dim
""").collect()[0][0]

In [ ]:
# new_users = new_users.withColumn("user_dim_key", ((F.monotonically_increasing_id()+1)+max_value)).display()
from pyspark.sql import Window
window_spec = Window.orderBy(["start_date","lot_number"])

new_lots = new_lots.withColumn(
    "lot_dim_key",
    (F.row_number().over(window_spec) + F.lit(max_value))
)

In [ ]:
final_lots = old_lots.union(new_lots)

---
## Step 6 · MERGE into Gold Layer

`old_lots` (existing, with keys) and `new_lots` (newly keyed) are unioned into `final_lots` and registered as a Spark temp view.

The `MERGE INTO company_db.gold.dim_lot` applies two operations atomically on the composite key `auction_id + lot_number + start_date`:

| Condition | Action | Notes |
|---|---|---|
| MATCHED | **UPDATE** all lot fields + `updated_at = current_timestamp()` | Keeps item details, status, and category in sync as auction data is refreshed |
| NOT MATCHED | **INSERT** full row with `created_at = current_timestamp()`, `updated_at = current_timestamp()` | Adds new lots |

> `lot_dim_key` is only written on **INSERT** — not updated on MATCH — to preserve the stable surrogate key.

In [ ]:
final_lots

In [ ]:
final_lots.createOrReplaceTempView("final_lots")

In [ ]:

spark.sql(
f"""
MERGE INTO {catalog}.gold.lots_dim AS target
USING final_lots AS source
ON target.auction_id = source.auction_id
   AND target.lot_number = source.lot_number
   AND target.start_date = source.start_date
WHEN MATCHED THEN
UPDATE SET
  target.lot_id = source.lot_id,
  target.item_name = source.item_name,
  target.link = source.link,
  target.exportedimgs = source.exportedimgs,
  target.format = source.format,
  target.`attribute:country` = source.`attribute:country`,
  target.`attribute:region` = source.`attribute:region`,
  target.region = source.region,
  target.country = source.country,
  target.item_year = source.item_year,
  target.lot_category = source.lot_category,
  target.status = source.status,
  target.start_date = source.start_date,
  target.end_date = source.end_date,
  target.auction_category = source.auction_category,
  target.updated_at = current_timestamp()
WHEN NOT MATCHED THEN
INSERT (
  lot_dim_key, auction_id, lot_id, lot_number, item_name, link, exportedimgs,
  format, `attribute:country`, `attribute:region`, region, country, item_year,
  lot_category, status, start_date, end_date, auction_category,
  created_at, updated_at
)
VALUES (
  source.lot_dim_key, source.auction_id, source.lot_id, source.lot_number,
  source.item_name, source.link, source.exportedimgs, source.format,
  source.`attribute:country`, source.`attribute:region`, source.region,
  source.country, source.item_year, source.lot_category, source.status,
  source.start_date, source.end_date, source.auction_category,
  current_timestamp(), current_timestamp()
)""")

---
## Step 7 · Verification

Queries the final `company_db.gold.dim_lot` table to confirm rows were correctly inserted or updated. Spot-check that:
- `lot_dim_key` values are sequential with no gaps for new lots
- `created_at` is populated for all rows
- `updated_at` reflects the current run timestamp for modified rows
- No lots are missing `auction_category` (would indicate a failed join with the sales report)

In [ ]:

# Verify MERGE output: row counts, surrogate key range, and data quality checks

spark.sql(f"""
SELECT
  COUNT(*)                                                          AS total_lots,
  COUNT(DISTINCT auction_id)                                        AS unique_auctions,
  MIN(lot_dim_key)                                                  AS min_lot_dim_key,
  MAX(lot_dim_key)                                                  AS max_lot_dim_key,
  SUM(CASE WHEN auction_category IS NULL THEN 1 END)                AS missing_auction_category,
  SUM(CASE WHEN item_name IS NULL THEN 1 END)                       AS missing_item_name,
  COUNT(DISTINCT country)                                           AS unique_countries,
  COUNT(DISTINCT item_year)                                           AS unique_item_years,
  MAX(updated_at)                                                   AS latest_updated
FROM {catalog}.gold.lots_dim;""")

In [ ]:
# %sql
# delete from company_db.gold.lots_dim
# where start_date = '2025-03-18'

In [ ]:
# %sql
# DESCRIBE HISTORY company_db.gold.lots_dim;

In [ ]:
# %sql
# SELECT * FROM company_db.gold.lots_dim VERSION AS OF 1
# where start_date = '2025-03-18';

In [ ]:
# %sql
# RESTORE TABLE company_db.gold.lots_dim TO VERSION AS OF 1;